# Superstore - Retail

#### Data pre-processing

In [ ]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")
data.shape

(9994, 21)

In [3]:
data.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [4]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
data.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [6]:
# Creating unique data tables
data["product_id"] = [x.split("-")[2] for x in data["Product ID"].values]
customer = data[["Customer ID", "Customer Name", "Segment", "Country","Region", "State", "City", "Postal Code", ]].drop_duplicates().reset_index(drop=True)
product = data[["product_id","Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year
dates['quarter'] = dates['date'].dt.quarter


product_price_history = data[["product_id", "Product ID","Category","Sub-Category", "Discount", "Sales","Quantity",
        "Profit"]].drop_duplicates().merge(orders, 
                        on="Product ID").merge(
                            dates, 
                            left_on="Order Date", 
                            right_on="date")[["product_id", "Category","Sub-Category",
                                                "year","month_num", "month",
                                                "Discount_x",
                                                "Sales_x","Quantity_x","Profit_x"]].drop_duplicates().sort_values(
                     ["year","month_num", "product_id"]).reset_index(drop=True)


product_price = product_price_history[product_price_history["Discount_x"] == 0.0]
product_price["Sales_x"] = round(product_price["Sales_x"] / product_price_history["Quantity_x"],2)
product_price["Profit_x"] = round(product_price["Profit_x"] / product_price_history["Quantity_x"],2)
product_price["cost"] = round(product_price["Sales_x"] - product_price["Profit_x"],2)
product_price= product_price[["product_id", "Category","Sub-Category","year","month","cost"]].drop_duplicates().reset_index(drop=True)
product_price.columns = ["product_id","category","sub_category","year","month","cost"]

product_price_disc = product_price_history[product_price_history["Discount_x"] != 0.0]
product_price_disc["cost"] = round(product_price_disc["Sales_x"] - product_price_disc["Profit_x"],2)
product_price_disc= product_price_disc[["product_id","year","month", "Sales_x","Discount_x", "Profit_x","cost"]].drop_duplicates().reset_index(drop=True)
product_price_disc.columns = ["product_id","year","month","revenue", "discount", "profit","cost"]



In [7]:
# creating aggregated view of orders
columns = ['Customer ID', 'Order ID', 'Order Date', 'year',"quarter",  'month', "month_num",'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Customer ID", "Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","quarter", "month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg = orders_agg.sort_values('Order Date').reset_index(drop=True)
orders_agg.columns = ["customer_id", 'order_id', 'order_date', 'year',"quarter",'month', "month_num",'ship_mode', 'days_till_shipped', 'revenue',
       'quantity', 'products' ]
orders_agg

,customer_id,order_id,order_date,year,quarter,month,month_num,ship_mode,days_till_shipped,revenue,quantity,products
0,BD-11500,CA-2014-140795,2014-01-02,2014,1,January,1,First Class,59,468.900,6,1
1,GW-14605,CA-2014-168312,2014-01-03,2014,1,January,1,Standard Class,181,513.861,6,2
2,VF-21715,CA-2014-113880,2014-01-03,2014,1,January,1,Standard Class,120,651.588,9,2
3,HR-14770,US-2014-143707,2014-01-03,2014,1,January,1,Standard Class,120,5.940,3,1
4,DB-13060,CA-2014-104269,2014-01-03,2014,1,January,1,Second Class,151,457.568,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5004,MC-17845,US-2017-102638,2017-12-29,2017,4,December,12,First Class,2,6.030,3,1
5005,CC-12430,CA-2017-126221,2017-12-30,2017,4,December,12,Standard Class,122,209.300,2,1
5006,JM-15580,CA-2017-156720,2017-12-30,2017,4,December,12,Standard Class,61,3.024,3,1
5007,PO-18865,CA-2017-143259,2017-12-30,2017,4,December,12,Standard Class,61,466.842,14,3


In [8]:
# Let's check total  orders
print("Total orders:", len(orders_agg['order_id']))

# Let's check total  sales
print("Total sales:", round(orders_agg['revenue'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793
